# Autoencoders — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def normal_pdf(x,mu,sigma):
    return np.exp(-0.5*((x-mu)/sigma)**2)/(sigma*np.sqrt(2*np.pi))
def softmax(a):
    a=np.asarray(a,dtype=float); e=np.exp(a-a.max()); return e/e.sum()
def mse(a,b): return float(np.mean((np.asarray(a)-np.asarray(b))**2))
print('setup ok')

## a bottleneck reconstruction map
Lessons on linear algebra and optimization supply the encoder matrix, decoder matrix, and reconstruction loss; representation learning from earlier neural-network lessons explains why the hidden code can become a useful feature. This starts Part 10 by turning unsupervised structure into a compact latent space, which VAEs (10.2), flows (10.9), and latent diffusion (10.14) will make explicitly generative.

In [ ]:
# 1) bottleneck projection reconstructs points on the learned direction
X=np.array([[2.,1.],[1.5,1.2],[-1.,-.4],[-1.5,-1.]])
w=np.array([.8,.6]); Z=X@w; Xh=np.outer(Z,w)
plt.figure(figsize=(4,4)); plt.scatter(X[:,0],X[:,1],label='x'); plt.scatter(Xh[:,0],Xh[:,1],label='recon'); plt.axis('equal'); plt.legend(); plt.title('1D bottleneck reconstructions')
assert abs(Z[0]-2.2)<1e-9 and mse(X[0],Xh[0])<0.08

In [ ]:
# 2) turning the bottleneck direction changes reconstruction loss
thetas=np.linspace(0,math.pi,80); losses=[]
for th in thetas:
    ww=np.array([math.cos(th),math.sin(th)]); losses.append(mse(X,np.outer(X@ww,ww)))
plt.figure(figsize=(5,3)); plt.plot(thetas,losses); plt.xlabel('direction angle'); plt.ylabel('MSE'); plt.title('best bottleneck aligns with data')
assert min(losses)<max(losses)

In [ ]:
# 3) a too-wide identity has zero loss but learns no compression
Xh_id=X.copy(); Xh_narrow=Xh
plt.figure(figsize=(4,4)); plt.scatter(X[:,0],X[:,1],label='input'); plt.scatter(Xh_id[:,0],Xh_id[:,1],marker='x',label='wide identity'); plt.axis('equal'); plt.legend(); plt.title('wide code can copy')
assert mse(X,Xh_id)==0.0 and mse(X,Xh_narrow)>0

In [ ]:
# 4) feature scaling can dominate squared reconstruction loss
Xs=X.copy(); Xs[:,0]*=10; Zs=Xs@w; Xhs=np.outer(Zs,w)
plt.figure(figsize=(4,4)); plt.scatter(Xs[:,0],Xs[:,1]); plt.scatter(Xhs[:,0],Xhs[:,1]); plt.title('scaled coordinate dominates loss')
assert mse(Xs,Xhs)>mse(X,Xh)

In [ ]:
# 5) sampling latent codes decodes along the learned line
zgrid=np.linspace(-2.5,2.5,30); samples=np.outer(zgrid,w)
plt.figure(figsize=(4,4)); plt.plot(samples[:,0],samples[:,1]); plt.scatter(X[:,0],X[:,1]); plt.axis('equal'); plt.title('latent samples decode on a line')
assert samples.shape==(30,2)